# 缓存

QuantStudio 的缓存系统为计算图中的数据持久化提供支持。缓存以键值对形式存储 DataFrame 数据，支持通用数据缓存和时序数据缓存两种模式，底层基于文件系统实现，并通过文件锁保证多进程并发安全。

## 类层次

```
Cache (抽象基类, 键值对缓存)
├── DTCache (时序数据缓存, 带时点范围追踪)
│   ├── FileDTCache (基于文件的实现, 通用数据 + 时点数据)
│   │   └── FeatherDTCache (Apache Feather 格式)
│   └── FactorCache (因子数据缓存, 原始数据 + 因子数据)
│       └── FileFactorCache (基于文件的因子缓存)
│           └── FeatherFactorCache (Feather 格式因子缓存)
```

其中 `Cache`、`DTCache`、`FileDTCache`、`FeatherDTCache` 位于 `QuantStudio.Core`，`FactorCache` 及其子类位于 `QuantStudio.Factor`。

In [1]:
import warnings
warnings.filterwarnings(action="ignore")
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [2]:
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

---

## Cache — 缓存基类

`Cache` 是所有缓存的抽象基类，继承自 `__QS_Object__`。定义了缓存的核心接口：启动/结束生命周期、增删查数据项、状态持久化。

### 参数

In [4]:
from QuantStudio.Core.Cache import Cache

display(Markdown(Cache().Args.info()))

* StartMode(启动模式): typing.Literal['new', 'continue'], 默认值 'new', 启动缓存的方式: new 表示清空已有数据重新构造缓存；continue 表示通过 load 恢复之前的缓存状态, 当前取值: 'new'

### 生命周期

缓存通过 `start()` / `end()` 管理生命周期，支持上下文管理器协议 (`with` 语句):

```python
with Cache() as cache:
    cache.writeData(...)
    data = cache.readData(...)
# 退出 with 块时自动调用 end()
```

**启动模式 (StartMode):**

| 值 | 行为 |
|------|------|
| `"new"` | 清空已有数据，重新构造缓存 |
| `"continue"` | 通过 `load()` 恢复之前的缓存状态 |

**生命周期流程:**

```
start()
├── StartMode == "new"     → clearData()
└── StartMode == "continue" → load()

end(clear=False)  → dump()    # 持久化当前状态
end(clear=True)   → clearData()  # 清空并退出
```

In [5]:
print("=== Cache 核心方法 ===\n")

print("--- start() ---")
print(qs_help(Cache.start))

print("--- end() ---")
print(qs_help(Cache.end))

print("--- dump() ---")
print(qs_help(Cache.dump))

print("--- load() ---")
print(qs_help(Cache.load))

=== Cache 核心方法 ===

--- start() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.start(self)
说明文档:
    初始化缓存
--- end() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.end(self, clear=False)
说明文档:
    结束缓存
--- dump() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.dump(self)
说明文档:
    暂存缓存状态
--- load() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.load(self)
说明文档:
    恢复缓存状态


In [6]:
print("=== 数据操作方法 ===\n")

print("--- checkDataExistence() ---")
print(qs_help(Cache.checkDataExistence))

print("--- writeData() ---")
print(qs_help(Cache.writeData))

print("--- readData() ---")
print(qs_help(Cache.readData))

print("--- clearData() ---")
print(qs_help(Cache.clearData))

=== 数据操作方法 ===

--- checkDataExistence() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.checkDataExistence(self, key: str, create_if_not_exists: bool = False) -> bool
说明文档:
    检查某个数据项是否存在
    
    Args:
        key: 数据项的键
        create_if_not_exists: 如果不存在是否要创建该数据项
    
    Returns:
        是否存在该项数据
--- writeData() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.writeData(self, key: str, data: pandas.core.frame.DataFrame, if_exists: Literal['append', 'replace'] = 'append', data_type: Optional[Literal['double', 'string', 'object']] = None, meta: dict = {})
说明文档:
    写入数据
    
    Args:
        key: 数据项的键
        data: 待写入的数据
        if_exists: 如果该数据已经存在的更新方式, append 表示只添加新增的数据, replace 表示替换已有数据
        data_type: 写入数据的数据类型, None 表示让系统自己判断
        meta: 数据项的元信息
--- readData() ---
类型: function
模块: QuantStudio.Core.Cache
签名: Cache.readData(self, key: str, data_type: Optional[Literal['double', 'string', 'object']] = None) -> None | pandas.core.frame.DataFrame
说明文档:
    读取数据
 

### 并行同步

`getUpdateData()` 和 `updateCache()` 用于多进程并行计算后的状态同步：

- **`getUpdateData()`**: 并发运行后返回需要同步的内容（字典）
- **`updateCache(update_data)`**: 接收同步数据并更新本地缓存状态

这两个方法在 `ParallelEngine` 和 `TreeEngine` (Process 模式) 中由框架自动调用。

---

## DTCache — 时序数据缓存

`DTCache` 继承自 `Cache`，专门处理索引为 `datetime` 的时序 DataFrame。它在基类基础上增加了**时点范围追踪**机制，能够精确记录每个 key 的已缓存时点区间，从而判断数据缺失范围。

### 参数

In [8]:
import datetime as dt
from QuantStudio.Core.Cache import DTCache

DTRuler = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(10)]
display(Markdown(DTCache(args={"DTRuler": DTRuler}).Args.info()))

* StartMode(启动模式): typing.Literal['new', 'continue'], 默认值 'new', 启动缓存的方式: new 表示清空已有数据重新构造缓存；continue 表示通过 load 恢复之前的缓存状态, 当前取值: 'new'
* DTRuler(时点标尺): typing.List[datetime.datetime], 无默认值, 当前运行计算时点标尺, 当前取值: [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0), '...', datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)], 共 10 个元素
* MinDTUnit(最小时间单位): <class 'datetime.timedelta'>, 默认值 1 day, 0:00:00, 当前取值: 1 day, 0:00:00

### 时点范围追踪

`DTCache` 的核心能力是记录每个数据项已缓存的时点范围 (`_CachedDTRange`)，并提供**区间合并**逻辑。

#### 内部数据结构

`_CachedDTRange` 是 `{key: DataFrame(columns=["StartDT", "EndDT"])}` 的字典。每个 key 对应一个 DataFrame，记录该数据项已被缓存的一个或多个**连续时点区间**。

#### 核心方法

- **`getDTRange(key, dt_range)`**: 给定查询时点范围，返回缓存中**缺失**的时点范围。返回 `None` 表示缓存已完全覆盖。
- **`updateDTRange(key, dt_range)`**: 将新的时点范围合并到缓存记录中，自动合并相邻/重叠区间。
- **`_mergeDTRange(cached_dt_range, i)`**: 内部方法，检查并合并索引 `i` 处的区间与其前后相邻区间。

In [9]:
print("=== DTCache 时点范围方法 ===\n")

print("--- getDTRange() ---")
print(qs_help(DTCache.getDTRange))

print("--- updateDTRange() ---")
print(qs_help(DTCache.updateDTRange))

=== DTCache 时点范围方法 ===

--- getDTRange() ---
类型: function
模块: QuantStudio.Core.Cache
签名: DTCache.getDTRange(self, key: str, dt_range: Tuple[datetime.datetime, datetime.datetime]) -> Optional[Tuple[datetime.datetime, datetime.datetime]]
说明文档:
    给定时点范围，获取缓存中缺失的时点范围
    
    Args:
        key: 数据项的键
        dt_range: (起始时间, 结束时间), 时点范围
    
    Returns:
        缓存中缺失的时点范围, (起始时间, 结束时间), 返回 None 表示当前的缓存已经覆盖了给定的时点范围
--- updateDTRange() ---
类型: function
模块: QuantStudio.Core.Cache
签名: DTCache.updateDTRange(self, key: str, dt_range: Tuple[datetime.datetime, datetime.datetime])
说明文档:
    更新缓存中数据的时点范围
    
    Args:
        key: 数据项的键
        dt_range: (起始时间, 结束时间), 时点范围


#### 区间查询与合并逻辑

`getDTRange` 将查询范围与已缓存区间比较，返回实际需要计算的缺失范围：

```
已缓存区间:     [===A===]       [===B===]
查询范围:    |----?----|
          → 返回 (查询起始, 查询结束)  # 完全未覆盖

查询范围:       |----?----|
          → 返回 None  # 完全被 A 覆盖

查询范围:  |------?------|
          → 返回 (A.End+1, B.Start-1)  # 跨区间，返回中间缺口
```

`updateDTRange` 同样处理四种情况（新区间在空档 / 被已有区间完全覆盖 / 跨区间 / 半覆盖），并通过 `_mergeDTRange` 自动合并相邻区间。

### 时序数据操作

除基类的 `writeData` / `readData` 外，`DTCache` 新增了专用于时序数据的读写方法。区别在于这些方法额外维护 `_CachedDTRange` 时点范围记录。

In [10]:
print("=== DTCache 时序数据操作方法 ===\n")

print("--- checkDTDataExistence() ---")
print(qs_help(DTCache.checkDTDataExistence))

print("--- writeDTData() ---")
print(qs_help(DTCache.writeDTData))

print("--- readDTData() ---")
print(qs_help(DTCache.readDTData))

print("--- clearDTData() ---")
print(qs_help(DTCache.clearDTData))

=== DTCache 时序数据操作方法 ===

--- checkDTDataExistence() ---
类型: function
模块: QuantStudio.Core.Cache
签名: DTCache.checkDTDataExistence(self, key: str, create_if_not_exists: bool = False) -> bool
说明文档:
    检查某个时点数据项是否存在
    
    Args:
        key: 数据项的键
        create_if_not_exists: 如果不存在是否要创建该数据项
    
    Returns:
        是否存在该项数据
--- writeDTData() ---
类型: function
模块: QuantStudio.Core.Cache
签名: DTCache.writeDTData(self, key: str, data: pandas.core.frame.DataFrame, if_exists: Literal['append', 'replace'] = 'append', data_type: Optional[Literal['double', 'string', 'object']] = None, meta: dict = {})
说明文档:
    写入时点数据
    
    Args:
        key: 数据项的键
        data: 待写入的数据
        if_exists: 如果该数据已经存在的更新方式, append 表示只添加新增的数据, replace 表示替换已有数据
        data_type: 写入数据的数据类型, None 表示让系统自己判断
        meta: 数据项的元信息
--- readDTData() ---
类型: function
模块: QuantStudio.Core.Cache
签名: DTCache.readDTData(self, key: str, data_type: Optional[Literal['double', 'string', 'object']] = None) -> None | pandas.core.

#### 时点范围追踪示例

下面通过具体例子展示时点范围的查询和更新逻辑：

In [12]:
import datetime as dt
import pandas as pd
from QuantStudio.Core.Cache import DTCache

# 构造一个 DTCache 实例用于演示
DTRuler = [dt.datetime(2025, 1, i) for i in range(1, 32)]  # 1月每天

class DemoDTCache(DTCache):
    """仅用于演示时点范围追踪的 DTCache 子类"""
    def checkDataExistence(self, key, create_if_not_exists=False):
        return key in self._CachedDTRange
    def writeData(self, key, data, if_exists="append", data_type=None, meta={}):
        pass
    def readData(self, key, data_type=None):
        return None
    def clearData(self, key=None):
        pass
    def checkDTDataExistence(self, key, create_if_not_exists=False):
        return key in self._CachedDTRange
    def writeDTData(self, key, data, if_exists="append", data_type=None, meta={}):
        pass
    def readDTData(self, key, data_type=None):
        return None
    def clearDTData(self, key=None):
        if key:
            self._CachedDTRange.pop(key, None)
        else:
            self._CachedDTRange = {}
    def dump(self):
        pass
    def load(self):
        pass

cache = DemoDTCache(args={"DTRuler": DTRuler, "MinDTUnit": dt.timedelta(1)})
cache.start()

# 步骤1: 查询 1月5日~1月10日，缓存为空 => 返回完整查询范围
missing = cache.getDTRange("test_key", (DTRuler[4], DTRuler[9]))
print(f"步骤1 - 首次查询缺失范围: {missing[0].date()} ~ {missing[1].date()}")

# 步骤2: 更新缓存，标记 1月5日~1月10日 已缓存
cache.updateDTRange("test_key", (DTRuler[4], DTRuler[9]))
print(f"步骤2 - 缓存区间: {cache._CachedDTRange['test_key'].to_dict('records')}")

# 步骤3: 再次查询相同范围 => 返回 None（已有缓存）
missing = cache.getDTRange("test_key", (DTRuler[4], DTRuler[9]))
print(f"步骤3 - 相同范围查询: {missing}  (None = 缓存已覆盖)")

# 步骤4: 查询扩展范围 1月3日~1月12日 => 返回两端的缺失部分
missing = cache.getDTRange("test_key", (DTRuler[2], DTRuler[11]))
if missing:
    print(f"步骤4 - 扩展范围查询缺失: {missing[0].date()} ~ {missing[1].date()}")

# 步骤5: 再缓存 1月1日~1月6日，区间应与已有区间合并
cache.updateDTRange("test_key", (DTRuler[0], DTRuler[5]))
print(f"步骤5 - 合并后缓存区间: {cache._CachedDTRange['test_key'].to_dict('records')}")

cache.end(clear=True)

步骤1 - 首次查询缺失范围: 2025-01-05 ~ 2025-01-10
步骤2 - 缓存区间: [{'StartDT': Timestamp('2025-01-05 00:00:00'), 'EndDT': Timestamp('2025-01-10 00:00:00')}]
步骤3 - 相同范围查询: None  (None = 缓存已覆盖)
步骤4 - 扩展范围查询缺失: 2025-01-03 ~ 2025-01-12
步骤5 - 合并后缓存区间: [{'StartDT': Timestamp('2025-01-01 00:00:00'), 'EndDT': Timestamp('2025-01-10 00:00:00')}]


### 并行同步增强

`DTCache` 覆写了 `getUpdateData()` 和 `updateCache()`，支持按 key 列表选择性同步 `_CachedDTRange`：

- `getUpdateData(key_list=None)`: 返回 `{"_CachedDTRange": {...}}`，可指定只同步部分 key 的时点范围
- `updateCache(update_data)`: 将其他进程的时点范围合并到本地，调用 `updateDTRange` 自动做区间合并

---

## FileDTCache — 基于文件的时序缓存

`FileDTCache` 是 `DTCache` 的第一个具体实现，将缓存数据存储在文件系统中。它使用 `FileLock` 保证多进程并发写入安全，并通过 pickle 实现缓存状态的持久化。

### 参数

In [13]:
from QuantStudio.Core.FileCache import FileDTCache

display(Markdown(FileDTCache(args={"DTRuler": DTRuler}).Args.info()))

* StartMode(启动模式): typing.Literal['new', 'continue'], 默认值 'new', 启动缓存的方式: new 表示清空已有数据重新构造缓存；continue 表示通过 load 恢复之前的缓存状态, 当前取值: 'new'
* DTRuler(时点标尺): typing.List[datetime.datetime], 无默认值, 当前运行计算时点标尺, 当前取值: [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0), '...', datetime.datetime(2025, 1, 30, 0, 0), datetime.datetime(2025, 1, 31, 0, 0)], 共 31 个元素
* MinDTUnit(最小时间单位): <class 'datetime.timedelta'>, 默认值 1 day, 0:00:00, 当前取值: 1 day, 0:00:00
* CacheDir(缓存目录): typing.Optional[typing.Annotated[pathlib.Path, PathType(path_type='dir')]], 默认值 None, 当前取值: None
* StateFile(状态文件): <class 'pathlib.Path'>, 默认值 'state.pkl', 用于存储缓存的状态, 当前取值: 'state.pkl'
* Suffix(后缀): <class 'str'>, 默认值 '', 当前取值: ''

### 目录结构

```
CacheDir/
├── LockFile              # 文件锁（跨进程互斥）
├── state.pkl             # 缓存状态持久化（dump/load）
├── Data/                 # 通用数据存放目录
│   ├── <key>.lock        # 每个 key 的文件锁
│   ├── <key><suffix>     # 数据文件
│   └── <key>_meta.json   # 元信息（可选）
├── DTData/               # 时点数据存放目录
│   ├── <key>.lock
│   ├── <key><suffix>
│   └── <key>_meta.json
└── _Cache.lock           # 全局缓存锁（清理时使用）
```

如果指定的 `CacheDir` 不存在，自动回退到系统临时目录 (`tempfile.TemporaryDirectory`)。

### 启动流程

```
start()
├── 定位/创建 CacheDir（不存在则回退临时目录）
├── 设置 DataDir、DTDataDir 子目录
├── 根据 StartMode:
│   ├── "new"     → clearData() + clearDTData()
│   └── "continue" → load() 恢复之前的 _CachedDTRange
├── 确保 Data/ 和 DTData/ 目录存在
└── 标记 _isStarted = True
```

### 数据读写

`FileDTCache` 实现了所有抽象方法。读写操作均通过 `FileLock` 加锁：

```python
# writeData 内部
with FileLock(data_dir + os.sep + key + ".lock") as DataLock:
    self.writeDataFrame(path=..., data=data, ...)
    if meta:
        self.writeMeta(path=..., meta=meta)
```

`writeDataFrame` 和 `readDataFrame` 是抽象方法，留给子类实现具体的存储格式。

### 状态持久化

- **`dump()`**: 将 `_CachedDTRange` 等状态序列化到 `state.pkl`
- **`load()`**: 从 `state.pkl` 反序列化恢复状态

这使得 `"continue"` 模式下可以跨会话恢复缓存。

In [14]:
print("=== FileDTCache 关键方法 ===\n")

print("--- start() ---")
print(qs_help(FileDTCache.start))

print("--- dump() ---")
print(qs_help(FileDTCache.dump))

print("--- load() ---")
print(qs_help(FileDTCache.load))

print("--- writeData() ---")
print(qs_help(FileDTCache.writeData))

print("--- readData() ---")
print(qs_help(FileDTCache.readData))

print("--- clearData() ---")
print(qs_help(FileDTCache.clearData))

=== FileDTCache 关键方法 ===

--- start() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FileDTCache.start(self)
说明文档:
    初始化缓存
--- dump() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FileDTCache.dump(self)
说明文档:
    暂存缓存状态
--- load() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FileDTCache.load(self)
说明文档:
    恢复缓存状态
--- writeData() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FileDTCache.writeData(self, key: str, data: pandas.core.frame.DataFrame, if_exists: Literal['append', 'replace'] = 'append', data_type: Optional[Literal['double', 'string', 'object']] = None, meta: dict = {})
说明文档:
    写入数据
    
    Args:
        key: 数据项的键
        data: 待写入的数据
        if_exists: 如果该数据已经存在的更新方式, append 表示只添加新增的数据, replace 表示替换已有数据
        data_type: 写入数据的数据类型, None 表示让系统自己判断
        meta: 数据项的元信息
--- readData() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FileDTCache.readData(self, key: str, data_type: Optional[Literal['double', 'string', 'object']] = None) -> None 

---

## FeatherDTCache — Feather 格式缓存

`FeatherDTCache` 继承自 `FileDTCache`，使用 [Apache Feather](https://arrow.apache.org/docs/python/feather.html) 格式存储 DataFrame。Feather 是列式二进制格式，读写速度远快于 CSV/pickle，适合大规模量化数据的存取。

### 特点

- 默认文件后缀 `.feather`
- 对 `data_type="object"` 的数据回退到 pickle 格式（Feather 不完全支持 object 类型）
- `readDataFrame` 中捕获 `ArrowInvalid` 异常（当文件损坏或格式不兼容时返回 `None`）

In [15]:
from QuantStudio.Core.FileCache import FeatherDTCache

print("=== FeatherDTCache ===\n")

print("--- writeDataFrame() ---")
print(qs_help(FeatherDTCache.writeDataFrame))

print("--- readDataFrame() ---")
print(qs_help(FeatherDTCache.readDataFrame))

=== FeatherDTCache ===

--- writeDataFrame() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FeatherDTCache.writeDataFrame(self, path: str, data: pandas.core.frame.DataFrame, if_exists: Literal['append', 'replace'] = 'replace', ignore_index: bool = True, data_type: Optional[Literal['double', 'string', 'object']] = None)
说明文档:
    ⚠️  未找到文档字符串（包括父类）
    该对象可能：
    - 是内置函数/方法（C 实现，无 __doc__）
    - 确实没有文档
--- readDataFrame() ---
类型: function
模块: QuantStudio.Core.FileCache
签名: FeatherDTCache.readDataFrame(self, path: str, data_type: Optional[Literal['double', 'string', 'object']] = None) -> None | pandas.core.frame.DataFrame
说明文档:
    ⚠️  未找到文档字符串（包括父类）
    该对象可能：
    - 是内置函数/方法（C 实现，无 __doc__）
    - 确实没有文档


### 使用示例

In [16]:
import tempfile, os, datetime as dt
import numpy as np
import pandas as pd
from QuantStudio.Core.FileCache import FeatherDTCache

# 创建临时缓存目录
tmp_dir = tempfile.mkdtemp()

DTRuler = [dt.datetime(2025, 1, i) for i in range(1, 11)]  # 1月1日~10日

# 创建 FeatherDTCache 实例
cache = FeatherDTCache(args={
    "Suffix": ".feather",
    "CacheDir": tmp_dir,
    "DTRuler": DTRuler,
    "MinDTUnit": dt.timedelta(1),
    "StartMode": "new"
})

with cache:
    # 写入通用数据
    df1 = pd.DataFrame(np.random.randn(5, 3), columns=["A", "B", "C"])
    cache.writeData("general_data", df1)
    print(f"写入通用数据: shape={df1.shape}")
    
    # 读取通用数据
    result = cache.readData("general_data")
    print(f"读取通用数据: shape={result.shape}")
    
    # 写入时序数据（带 datetime index）
    dt_index = DTRuler[:5]
    df2 = pd.DataFrame(np.random.randn(5, 3), index=dt_index, columns=["X", "Y", "Z"])
    cache.writeDTData("ts_data", df2)
    print(f"写入时序数据: shape={df2.shape}, index={df2.index[0].date()}~{df2.index[-1].date()}")
    
    # 读取时序数据
    result = cache.readDTData("ts_data")
    print(f"读取时序数据: shape={result.shape}")

# 清理
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)
print("\n临时目录已清理")

2026-06-26 15:08:59,971 | QS | WARNING : 找不到配置文件: C:\Users\lenovo\QuantStudioConfig\FeatherDTCacheConfig.json


写入通用数据: shape=(5, 3)
读取通用数据: shape=(5, 3)
写入时序数据: shape=(5, 3), index=2025-01-01~2025-01-05
读取时序数据: shape=(5, 3)

临时目录已清理


---

## FactorCache — 因子数据缓存

`FactorCache` 位于 `QuantStudio.Factor` 模块，继承自 `DTCache`，专为因子计算场景设计。在通用缓存和时序缓存的基础上，增加了**多进程空间管理**和**原始数据/因子数据分离存储**。

### 参数

In [17]:
from QuantStudio.Factor.FactorCache import FactorCache

display(Markdown(FactorCache(args={"DTRuler": DTRuler}).Args.info()))

* StartMode(启动模式): typing.Literal['new', 'continue'], 默认值 'new', 启动缓存的方式: new 表示清空已有数据重新构造缓存；continue 表示通过 load 恢复之前的缓存状态, 当前取值: 'new'
* DTRuler(时点标尺): typing.List[datetime.datetime], 无默认值, 当前运行计算时点标尺, 当前取值: [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 1, 2, 0, 0), datetime.datetime(2025, 1, 3, 0, 0), '...', datetime.datetime(2025, 1, 9, 0, 0), datetime.datetime(2025, 1, 10, 0, 0)], 共 10 个元素
* MinDTUnit(最小时间单位): <class 'datetime.timedelta'>, 默认值 1 day, 0:00:00, 当前取值: 1 day, 0:00:00
* PIDs(进程ID): list[str], 默认值 ['0'], 当前取值: ['0']

### 数据分层

`FactorCache` 将缓存分为四个层次：

| 层次 | 目录 | 方法 | 说明 |
|------|------|------|------|
| 通用数据 | `Data/` | `writeData` / `readData` | 继承自 Cache |
| 时序数据 | `DTData/` | `writeDTData` / `readDTData` | 继承自 DTCache |
| 原始数据 | `RawData/<PID>/` | `writeRawData` / `readRawData` | 按进程分目录，支持按 ID 列切分 |
| 因子数据 | `FactorData/<PID>/` | `writeFactorData` / `readFactorData` | 按进程分目录，支持等待同步 |

### 多进程支持

- **`PIDs`**: 进程 ID 列表，每个进程在 `RawData/` 和 `FactorData/` 下有独立子目录
- **`writeRawData`**: 按 `id_col` 列将数据切分后分发到各进程目录
- **`readFactorData`** 的 `wait` 参数: 当其他进程数据未就绪时，轮询等待直到数据完整

### 具体实现类

- **`FileFactorCache(FileDTCache, FactorCache)`**: 基于文件的因子缓存，使用 pickle 格式
- **`FeatherFactorCache(FileFactorCache, FeatherDTCache)`**: 基于 Feather 格式的因子缓存，默认后缀 `.feather`

In [18]:
print("=== FactorCache 核心方法 ===\n")

print("--- writeRawData() ---")
print(qs_help(FactorCache.writeRawData))

print("--- readRawData() ---")
print(qs_help(FactorCache.readRawData))

print("--- writeFactorData() ---")
print(qs_help(FactorCache.writeFactorData))

print("--- readFactorData() ---")
print(qs_help(FactorCache.readFactorData))

=== FactorCache 核心方法 ===

--- writeRawData() ---
类型: function
模块: QuantStudio.Factor.FactorCache
签名: FactorCache.writeRawData(self, key: str, raw_data: Dict[str, pandas.core.frame.DataFrame], id_col: str = 'ID', if_exists: Literal['append', 'replace'] = 'append', pid_ids: Optional[Dict[str, List[str]]] = None, meta: dict = {})
说明文档:
    写入原始数据
    
    Args:
        key: 数据项的键
        raw_data: 待写入的原始数据, {field: DataFrame}
        id_col: ID 所在的列名, 如果 ID 列存在则按照 ID 切分后存入各个进程空间, 否则则将所有数据分别都存入各个进程空间
        if_exists: 如果该数据已经存在的更新方式, append 表示只添加新增的数据, replace 表示替换已有数据
        pid_ids: 进程及其分配的 ID 序列, {进程ID: [ID]}
        meta: 数据项的元信息
--- readRawData() ---
类型: function
模块: QuantStudio.Factor.FactorCache
签名: FactorCache.readRawData(self, key: str, target_fields: Optional[List[str]] = None, pids: Optional[List[str]] = None) -> Dict[str, pandas.core.frame.DataFrame]
说明文档:
    读取原始数据
    
    Args:
        key: 数据项的键
        target_fields: 待读取的字段列表, None 表示所有的字段
        pids: 待读取的进程 ID 列表
   

---

## 缓存与计算图集成

缓存在计算图中的典型使用模式：

1. **创建 Context 并绑定缓存**: `context.DataCache = cache`
2. **init_compute 阶段**: 节点通过 `getDTRange` 判断数据缺失范围，注册准备任务
3. **prepare_compute 阶段**: 引擎调用节点的 `prepare_compute`，在此阶段从数据库读取数据并写入缓存
4. **compute 阶段**: 节点从缓存读取数据参与计算
5. **并行计算后**: 通过 `getUpdateData()` / `updateCache()` 同步各进程的缓存状态

```python
# 典型使用模式
from QuantStudio.Core.Node import Context
from QuantStudio.Core.FileCache import FeatherDTCache
from QuantStudio.Core.CalcEngine import Engine

cache = FeatherDTCache(args={
    "CacheDir": "/path/to/cache",
    "DTRuler": dt_ruler,
    "StartMode": "continue"  # 复用已有缓存
})

context = Context()
context.DataCache = cache  # 绑定缓存到上下文

with cache:  # 启动缓存（或手动 cache.start()）
    results = engine.run(node_list, context)
# 退出时自动 dump 状态
```

---

## 总结

| 类 | 模块 | 层级 | 职责 |
|------|------|------|------|
| `Cache` | `Core.Cache` | 抽象基类 | 定义键值对缓存接口和生命周期 |
| `DTCache` | `Core.Cache` | 抽象类 | 增加时点范围追踪和区间合并逻辑 |
| `FileDTCache` | `Core.FileCache` | 具体类 | 基于文件的实现，FileLock 并发安全 |
| `FeatherDTCache` | `Core.FileCache` | 具体类 | Apache Feather 格式，高性能列式存储 |
| `FactorCache` | `Factor.FactorCache` | 抽象类 | 多进程因子缓存，原始/因子数据分离 |
| `FileFactorCache` | `Factor.FactorCache` | 具体类 | 基于文件的因子缓存 |
| `FeatherFactorCache` | `Factor.FactorCache` | 具体类 | Feather 格式因子缓存 |

### 选择指南

- **简单键值存储**：使用 `FileDTCache` 或 `FeatherDTCache`
- **需要时点范围追踪**：必须使用 `DTCache` 子类
- **因子研究场景**：使用 `FactorCache` 子类（支持多进程、原始数据与因子数据分离）
- **高性能需求**：优先选择 `FeatherDTCache` / `FeatherFactorCache`（Feather 列式存储读写更快）
- **跨会话复用**：设置 `StartMode="continue"`，缓存状态通过 `state.pkl` 持久化